# ARIMA Experiment

This notebook improves the aggregate ARIMA baseline without using ARIMAX or SARIMA. It tests:

1. small ARIMA order search;
2. last-year vs blended allocation from weekly total forecast to Store-Dept rows.

The baseline result to beat is:

```text
ARIMA(1,1,1) validation WMAE = 1856.86053
Seasonal naive validation WMAE = 1800.17359
```

ARIMAX with external regressors is kept in `model_experiment_ARIMAX.ipynb`.


In [1]:
%pip install -q "numpy>=1.24,<3" "pandas>=2.0,<3" "scikit-learn>=1.3,<2" "statsmodels>=0.14,<1" "wandb>=0.19,<1" "cloudpickle>=3.0,<4"

from pathlib import Path
import itertools
import warnings
import cloudpickle

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

try:
    import wandb
except Exception:
    wandb = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

In [2]:
try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

Mounted at /content/drive


## Configuration

The search is intentionally small. This notebook is pure ARIMA: no external regressors and no SARIMA seasonal order.


In [3]:
DATA_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/walmart_competition_data"),
    Path("/content/Walmart-Recruiting---Store-Sales-Forecasting/data"),
    Path("../../../../data"),
    Path("../../../data"),
    Path("data"),
]

VALIDATION_WEEKS = 39
HOLIDAY_WEIGHT = 5.0
ORDER_GRID = list(itertools.product([0, 1, 2], [0, 1], [0, 1, 2]))
ALLOCATION_STRATEGIES = ["last_year_share", "blended_share"]

RUN_WANDB = True
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
WANDB_RUN_NAME = "ARIMA_Order_Allocation_Experiment"
WANDB_MODE = "online"

RUN_TEST_SUBMISSION = False
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def resolve_data_dir(candidates):
    required = ["train.csv", "test.csv", "features.csv", "stores.csv"]
    for candidate in candidates:
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError("Could not find Walmart data directory.")


def weighted_mae(y_true, y_pred, is_holiday, holiday_weight=5.0):
    weights = np.where(np.asarray(is_holiday).astype(bool), holiday_weight, 1.0)
    return float(
        np.sum(weights * np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        / np.sum(weights)
    )


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print(f"Using data directory: {DATA_DIR.resolve()}")

Using data directory: /content/drive/MyDrive/walmart_competition_data


In [4]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
features = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")

train = train.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
test = test.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
features = features.sort_values(["Date", "Store"]).reset_index(drop=True)

print(
    {
        "train": train.shape,
        "test": test.shape,
        "features": features.shape,
        "stores": stores.shape,
    }
)
display(train.head())

{'train': (421570, 5), 'test': (115064, 4), 'features': (8190, 12), 'stores': (45, 3)}


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,2,2010-02-05,50605.27,False
2,1,3,2010-02-05,13740.12,False
3,1,4,2010-02-05,39954.04,False
4,1,5,2010-02-05,32229.38,False


In [5]:
all_dates = np.array(sorted(train["Date"].unique()))
validation_dates = all_dates[-VALIDATION_WEEKS:]
validation_start = validation_dates[0]

train_part = train[train["Date"] < validation_start].copy()
val_part = train[train["Date"] >= validation_start].copy()

print(
    {
        "train_rows": len(train_part),
        "validation_rows": len(val_part),
        "train_end": train_part["Date"].max().date(),
        "validation_start": pd.Timestamp(validation_start).date(),
        "validation_end": val_part["Date"].max().date(),
    }
)

{'train_rows': 305982, 'validation_rows': 115588, 'train_end': datetime.date(2012, 1, 27), 'validation_start': datetime.date(2012, 2, 3), 'validation_end': datetime.date(2012, 10, 26)}


## ARIMA Improvement Scope

Pure ARIMA does not use tabular feature engineering. To improve the baseline without turning it into ARIMAX, this notebook focuses on two things:

- choosing a better `(p, d, q)` order;
- improving the allocation from aggregate weekly forecast to Store-Dept rows.

This keeps the experiment classical and easy to compare with the baseline.


In [6]:
def weekly_total_sales(frame):
    return (
        frame.groupby("Date", as_index=True)["Weekly_Sales"]
        .sum()
        .sort_index()
        .asfreq("W-FRI")
    )


weekly_sales_train = weekly_total_sales(train_part)
print(
    {
        "weekly_train_points": len(weekly_sales_train),
        "weekly_train_start": weekly_sales_train.index.min().date(),
        "weekly_train_end": weekly_sales_train.index.max().date(),
    }
)

{'weekly_train_points': 104, 'weekly_train_start': datetime.date(2010, 2, 5), 'weekly_train_end': datetime.date(2012, 1, 27)}


## Allocation Strategies

The aggregate ARIMA model forecasts total weekly sales. Kaggle needs row-level Store-Dept predictions, so we compare two allocation strategies:

- `last_year_share`: uses same Store-Dept sales 52 weeks ago;
- `blended_share`: combines last-year sales with recent pre-validation average sales.

In [7]:
def fit_forecast_arima(y_train, forecast_dates, order):
    model = ARIMA(
        y_train,
        order=order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    result = model.fit()
    forecast = result.forecast(steps=len(forecast_dates))
    forecast = pd.Series(np.asarray(forecast), index=forecast_dates).clip(lower=0.0)
    return result, forecast


def make_row_forecast(
    target_frame,
    history_frame,
    aggregate_forecast,
    strategy="last_year_share",
    recent_weeks=13,
):
    target = target_frame[["Store", "Dept", "Date", "IsHoliday"]].copy()
    history = history_frame[["Store", "Dept", "Date", "Weekly_Sales"]].copy()

    last_year = history.copy()
    last_year["Date"] = last_year["Date"] + pd.Timedelta(days=364)
    last_year = last_year.rename(columns={"Weekly_Sales": "last_year_sales"})

    series_mean = (
        history.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .mean()
        .rename(columns={"Weekly_Sales": "series_mean_sales"})
    )
    recent_cutoff = history["Date"].max() - pd.Timedelta(days=7 * recent_weeks)
    recent_mean = (
        history[history["Date"] > recent_cutoff]
        .groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .mean()
        .rename(columns={"Weekly_Sales": "recent_mean_sales"})
    )

    target = target.merge(last_year, on=["Store", "Dept", "Date"], how="left")
    target = target.merge(series_mean, on=["Store", "Dept"], how="left")
    target = target.merge(recent_mean, on=["Store", "Dept"], how="left")

    if strategy == "last_year_share":
        base = target["last_year_sales"].fillna(target["series_mean_sales"])
    elif strategy == "blended_share":
        last_year_base = target["last_year_sales"].fillna(target["series_mean_sales"])
        recent_base = target["recent_mean_sales"].fillna(target["series_mean_sales"])
        base = 0.70 * last_year_base + 0.30 * recent_base
    else:
        raise ValueError(f"Unknown allocation strategy: {strategy}")

    target["allocation_base"] = base.fillna(0.0).clip(lower=0.0)
    date_base_sum = target.groupby("Date")["allocation_base"].transform("sum")
    row_count = target.groupby("Date")["allocation_base"].transform("size")
    target["share"] = np.where(
        date_base_sum > 0, target["allocation_base"] / date_base_sum, 1.0 / row_count
    )
    target["aggregate_forecast"] = target["Date"].map(aggregate_forecast)
    target["prediction"] = (
        (target["aggregate_forecast"] * target["share"]).fillna(0.0).clip(lower=0.0)
    )
    return target["prediction"].to_numpy()


def make_seasonal_naive_forecast(target_frame, history_frame):
    target = target_frame[["Store", "Dept", "Date"]].copy()
    history = history_frame[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
    history["Date"] = history["Date"] + pd.Timedelta(days=364)
    history = history.rename(columns={"Weekly_Sales": "last_year_sales"})
    fallback = (
        history_frame.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .median()
        .rename(columns={"Weekly_Sales": "series_median_sales"})
    )
    target = target.merge(history, on=["Store", "Dept", "Date"], how="left")
    target = target.merge(fallback, on=["Store", "Dept"], how="left")
    return (
        target["last_year_sales"]
        .fillna(target["series_median_sales"])
        .fillna(0.0)
        .clip(lower=0.0)
        .to_numpy()
    )

In [8]:
weekly_sales = weekly_total_sales(train_part)
forecast_dates = list(pd.to_datetime(validation_dates))

seasonal_naive_pred = make_seasonal_naive_forecast(val_part, train_part)
seasonal_naive_wmae = weighted_mae(
    val_part["Weekly_Sales"], seasonal_naive_pred, val_part["IsHoliday"], HOLIDAY_WEIGHT
)
print(f"Seasonal naive WMAE: {seasonal_naive_wmae:.4f}")

wandb_run = None
trial_table = None
if RUN_WANDB:
    if wandb is None:
        raise ImportError(
            "wandb is not available. Re-run the install/import cell or install wandb."
        )
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        job_type="arima-experiment",
        mode=WANDB_MODE,
        reinit=True,
        config={
            "validation_weeks": VALIDATION_WEEKS,
            "holiday_weight": HOLIDAY_WEIGHT,
            "order_grid_size": len(ORDER_GRID),
            "allocation_strategies": ALLOCATION_STRATEGIES,
            "model_family": "ARIMA",
            "uses_exog": False,
            "seasonal_order": "disabled",
        },
    )
    trial_table = wandb.Table(
        columns=[
            "trial",
            "order",
            "allocation",
            "wmae",
            "mae",
            "rmse",
            "improvement_vs_seasonal_naive_pct",
        ]
    )

results = []
trial = 0
for order in ORDER_GRID:
    try:
        _, aggregate_forecast = fit_forecast_arima(weekly_sales, forecast_dates, order)
    except Exception as exc:
        print(f"Trial {trial:03d} failed for order={order}: {exc}")
        trial += 1
        continue

    for allocation in ALLOCATION_STRATEGIES:
        pred = make_row_forecast(val_part, train_part, aggregate_forecast, allocation)
        wmae_value = weighted_mae(
            val_part["Weekly_Sales"], pred, val_part["IsHoliday"], HOLIDAY_WEIGHT
        )
        mae_value = float(mean_absolute_error(val_part["Weekly_Sales"], pred))
        rmse_value = rmse(val_part["Weekly_Sales"], pred)
        improvement = 100.0 * (seasonal_naive_wmae - wmae_value) / seasonal_naive_wmae
        row = {
            "trial": trial,
            "order": order,
            "allocation": allocation,
            "validation/wmae": wmae_value,
            "validation/mae": mae_value,
            "validation/rmse": rmse_value,
            "improvement_vs_seasonal_naive_pct": improvement,
        }
        results.append(row)
        print(
            f"Trial {trial:03d} | order={order} allocation={allocation} | WMAE={wmae_value:.4f} | improvement={improvement:.2f}%"
        )
        if RUN_WANDB:
            wandb.log(
                {
                    "trial": trial,
                    "validation/wmae": wmae_value,
                    "validation/mae": mae_value,
                    "validation/rmse": rmse_value,
                    "improvement_vs_seasonal_naive_pct": improvement,
                    "baseline/seasonal_naive_wmae": seasonal_naive_wmae,
                },
                step=trial,
            )
            trial_table.add_data(
                trial,
                str(order),
                allocation,
                wmae_value,
                mae_value,
                rmse_value,
                improvement,
            )
    trial += 1

results_df = pd.DataFrame(results).sort_values("validation/wmae").reset_index(drop=True)
best_result = results_df.iloc[0].to_dict()

if RUN_WANDB:
    wandb.log({"arima_experiment/trials": trial_table})
    wandb.summary["best_validation_wmae"] = float(best_result["validation/wmae"])
    wandb.summary["best_validation_mae"] = float(best_result["validation/mae"])
    wandb.summary["best_validation_rmse"] = float(best_result["validation/rmse"])
    wandb.summary["best_order"] = str(best_result["order"])
    wandb.summary["best_allocation"] = best_result["allocation"]
    wandb.summary["seasonal_naive_wmae"] = float(seasonal_naive_wmae)
    wandb.finish()

display(results_df.head(10))
best_result

Seasonal naive WMAE: 1800.1736


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nmetr23 (kende23-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Trial 000 | order=(0, 0, 0) allocation=last_year_share | WMAE=1836.5913 | improvement=-2.02%
Trial 000 | order=(0, 0, 0) allocation=blended_share | WMAE=2002.8350 | improvement=-11.26%
Trial 001 | order=(0, 0, 1) allocation=last_year_share | WMAE=1835.7033 | improvement=-1.97%
Trial 001 | order=(0, 0, 1) allocation=blended_share | WMAE=2003.6289 | improvement=-11.30%
Trial 002 | order=(0, 0, 2) allocation=last_year_share | WMAE=1852.3582 | improvement=-2.90%
Trial 002 | order=(0, 0, 2) allocation=blended_share | WMAE=2023.9889 | improvement=-12.43%
Trial 003 | order=(0, 1, 0) allocation=last_year_share | WMAE=2941.1402 | improvement=-63.38%
Trial 003 | order=(0, 1, 0) allocation=blended_share | WMAE=3149.6627 | improvement=-74.96%
Trial 004 | order=(0, 1, 1) allocation=last_year_share | WMAE=2183.7088 | improvement=-21.31%
Trial 004 | order=(0, 1, 1) allocation=blended_share | WMAE=2425.3955 | improvement=-34.73%
Trial 005 | order=(0, 1, 2) allocation=last_year_share | WMAE=1846.0585 |

baseline/seasonal_naive_wmae,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
improvement_vs_seasonal_naive_pct,███▁▅████▂█▅███▂▆█
trial,▁▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇██
validation/mae,▁▁▁█▃▁▁▁▁▇▁▄▁▁▁▇▃▁
validation/rmse,▁▁▁█▃▁▁▁▁▇▁▄▁▁▁▇▃▁
validation/wmae,▁▁▁█▄▁▁▁▁▇▁▄▁▁▁▇▃▁
baseline/seasonal_naive_wmae,1800.17359
best_allocation,last_year_share
best_order,"(1, 0, 2)"
best_validation_mae,1840.65361
best_validation_rmse,3937.04537


,trial,order,allocation,validation/wmae,validation/mae,validation/rmse,improvement_vs_seasonal_naive_pct
0,8,"(1, 0, 2)",last_year_share,1829.879987,1840.653606,3937.045372,-1.650196
1,14,"(2, 0, 2)",last_year_share,1834.686858,1841.263788,3937.679490,-1.917219
2,1,"(0, 0, 1)",last_year_share,1835.703255,1840.755151,3936.428171,-1.973680
3,0,"(0, 0, 0)",last_year_share,1836.591341,1841.826620,3939.168254,-2.023013
4,5,"(0, 1, 2)",last_year_share,1846.058549,1864.597955,3993.862035,-2.548918
5,6,"(1, 0, 0)",last_year_share,1851.987398,1846.035879,3945.610174,-2.878267
6,2,"(0, 0, 2)",last_year_share,1852.358159,1845.488705,3944.490944,-2.898863
7,10,"(1, 1, 1)",last_year_share,1856.860525,1843.287631,3919.594370,-3.148970
8,7,"(1, 0, 1)",last_year_share,1870.417502,1852.162456,3957.194077,-3.902063
9,12,"(2, 0, 0)",last_year_share,1873.401519,1852.854139,3958.675226,-4.067826


{'trial': 8,
 'order': (1, 0, 2),
 'allocation': 'last_year_share',
 'validation/wmae': 1829.879987169844,
 'validation/mae': 1840.653606132595,
 'validation/rmse': 3937.0453719213524,
 'improvement_vs_seasonal_naive_pct': -1.6501961609039517}

## Register Best ARIMA Pipeline

Run this cell after the experiment loop. It refits the best ARIMA order on all available training data, packages the fitted aggregate model with allocation history, logs it as a W&B model artifact, and links it to the W&B Model Registry.


In [ ]:
required_objects = ["best_result", "train", "weekly_total_sales"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        "Run previous experiment cells first. Missing: " + ", ".join(missing_objects)
    )
if wandb is None:
    raise ImportError(
        "wandb is not available. Re-run the install/import cell or install wandb."
    )

REGISTER_ARIMA_MODEL = True
ARIMA_REGISTRY_TARGET = "wandb-registry-model/Walmart_ARIMA_Pipeline"
ARIMA_MODEL_ARTIFACT_NAME = "walmart-arima-best-pipeline"


class AggregateARIMAPipeline:
    """Raw-test ARIMA pipeline for W&B Model Registry."""

    def __init__(
        self,
        model_result,
        order,
        allocation_strategy,
        observed_history,
        validation_metrics,
        metadata,
    ):
        self.model_result = model_result
        self.order = tuple(order)
        self.allocation_strategy = allocation_strategy
        self.observed_history = observed_history.copy()
        self.validation_metrics = dict(validation_metrics)
        self.metadata = dict(metadata)

    def _aggregate_forecast(self, raw_df):
        dates = list(pd.to_datetime(sorted(raw_df["Date"].unique())))
        forecast = self.model_result.forecast(steps=len(dates))
        return pd.Series(np.asarray(forecast), index=dates).clip(lower=0.0)

    def _make_row_forecast(self, target_frame, aggregate_forecast, recent_weeks=13):
        target = target_frame[["Store", "Dept", "Date", "IsHoliday"]].copy()
        history = self.observed_history[
            ["Store", "Dept", "Date", "Weekly_Sales"]
        ].copy()
        history["Date"] = pd.to_datetime(history["Date"])

        last_year = history.copy()
        last_year["Date"] = last_year["Date"] + pd.Timedelta(days=364)
        last_year = last_year.rename(columns={"Weekly_Sales": "last_year_sales"})
        series_mean = (
            history.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
            .mean()
            .rename(columns={"Weekly_Sales": "series_mean_sales"})
        )
        recent_cutoff = history["Date"].max() - pd.Timedelta(days=7 * recent_weeks)
        recent_mean = (
            history[history["Date"] > recent_cutoff]
            .groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
            .mean()
            .rename(columns={"Weekly_Sales": "recent_mean_sales"})
        )
        target = target.merge(last_year, on=["Store", "Dept", "Date"], how="left")
        target = target.merge(series_mean, on=["Store", "Dept"], how="left")
        target = target.merge(recent_mean, on=["Store", "Dept"], how="left")
        if self.allocation_strategy == "last_year_share":
            base = target["last_year_sales"].fillna(target["series_mean_sales"])
        elif self.allocation_strategy == "blended_share":
            last_year_base = target["last_year_sales"].fillna(
                target["series_mean_sales"]
            )
            recent_base = target["recent_mean_sales"].fillna(
                target["series_mean_sales"]
            )
            base = 0.70 * last_year_base + 0.30 * recent_base
        else:
            raise ValueError(f"Unknown allocation strategy: {self.allocation_strategy}")
        target["allocation_base"] = base.fillna(0.0).clip(lower=0.0)
        date_base_sum = target.groupby("Date")["allocation_base"].transform("sum")
        row_count = target.groupby("Date")["allocation_base"].transform("size")
        target["share"] = np.where(
            date_base_sum > 0,
            target["allocation_base"] / date_base_sum,
            1.0 / row_count,
        )
        target["aggregate_forecast"] = target["Date"].map(aggregate_forecast)
        target["prediction"] = (
            (target["aggregate_forecast"] * target["share"]).fillna(0.0).clip(lower=0.0)
        )
        return target["prediction"].to_numpy()

    def predict(self, raw_df):
        frame = raw_df.copy()
        frame["Date"] = pd.to_datetime(frame["Date"])
        if "IsHoliday" not in frame.columns:
            frame["IsHoliday"] = False
        aggregate_forecast = self._aggregate_forecast(frame)
        return self._make_row_forecast(frame, aggregate_forecast)


if REGISTER_ARIMA_MODEL:
    best_order = tuple(best_result["order"])
    best_allocation = best_result["allocation"]
    full_weekly_sales = weekly_total_sales(train)
    full_model = ARIMA(
        full_weekly_sales,
        order=best_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit()

    registry_output_dir = OUTPUT_DIR / "arima_registry"
    registry_output_dir.mkdir(parents=True, exist_ok=True)
    pipeline_path = registry_output_dir / "arima_best_pipeline.pkl"

    validation_metrics = {
        "best_validation_wmae": float(best_result["validation/wmae"]),
        "best_validation_mae": float(best_result["validation/mae"]),
        "best_validation_rmse": float(best_result["validation/rmse"]),
        "seasonal_naive_wmae": float(seasonal_naive_wmae),
        "improvement_vs_seasonal_naive_pct": float(
            best_result["improvement_vs_seasonal_naive_pct"]
        ),
    }
    pipeline = AggregateARIMAPipeline(
        model_result=full_model,
        order=best_order,
        allocation_strategy=best_allocation,
        observed_history=train,
        validation_metrics=validation_metrics,
        metadata={
            "model_family": "ARIMA",
            "uses_exog": False,
            "seasonal_order": "disabled",
            "validation_weeks": VALIDATION_WEEKS,
            "refit_scope": "full_train",
        },
    )
    with pipeline_path.open("wb") as file:
        cloudpickle.dump(pipeline, file)

    registry_run = wandb.init(
        project=WANDB_PROJECT,
        name="ARIMA_Best_Model_Registry",
        job_type="model_registration",
        mode=WANDB_MODE,
        reinit=True,
        config={
            **validation_metrics,
            "best_order": str(best_order),
            "best_allocation": best_allocation,
            "registry_target": ARIMA_REGISTRY_TARGET,
        },
    )
    model_artifact = wandb.Artifact(
        name=ARIMA_MODEL_ARTIFACT_NAME,
        type="model",
        description="Best aggregate ARIMA pipeline with row-level Store-Dept allocation.",
        metadata={
            **validation_metrics,
            "model_family": "ARIMA",
            "order": str(best_order),
            "allocation_strategy": best_allocation,
            "registry_target": ARIMA_REGISTRY_TARGET,
        },
    )
    model_artifact.add_file(str(pipeline_path))
    logged_artifact = registry_run.log_artifact(
        model_artifact, aliases=["best", "latest"]
    )
    registry_run.link_artifact(
        logged_artifact,
        target_path=ARIMA_REGISTRY_TARGET,
        aliases=["best-arima", "latest", "champion"],
    )
    registry_run.summary.update(validation_metrics)
    registry_run.summary["registry_target"] = ARIMA_REGISTRY_TARGET
    registry_run.summary["pipeline_artifact"] = ARIMA_MODEL_ARTIFACT_NAME
    registry_run.finish()

    print("Registered best ARIMA pipeline in W&B Model Registry:")
    print(f"  artifact: {ARIMA_MODEL_ARTIFACT_NAME}")
    print(f"  registry: {ARIMA_REGISTRY_TARGET}")
    print(f"  validation WMAE: {validation_metrics['best_validation_wmae']:.4f}")

## Optional Best ARIMA Submission

Set `RUN_TEST_SUBMISSION=True` only after reviewing validation results. This refits the best aggregate ARIMA setup on all training dates and creates a Kaggle CSV.


In [ ]:
if RUN_TEST_SUBMISSION:
    best_order = tuple(best_result["order"])
    best_allocation = best_result["allocation"]

    full_weekly_sales = weekly_total_sales(train)
    test_dates = list(pd.to_datetime(sorted(test["Date"].unique())))

    _, aggregate_test_forecast = fit_forecast_arima(
        full_weekly_sales, test_dates, best_order
    )
    test_pred = make_row_forecast(test, train, aggregate_test_forecast, best_allocation)

    submission = pd.DataFrame(
        {
            "Id": test["Store"].astype(str)
            + "_"
            + test["Dept"].astype(str)
            + "_"
            + test["Date"].dt.strftime("%Y-%m-%d"),
            "Weekly_Sales": test_pred,
        }
    )
    submission_path = OUTPUT_DIR / "arima_experiment_submission.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Saved {submission_path} with {len(submission)} rows")
    display(submission.head())
else:
    print(
        "Submission skipped. Set RUN_TEST_SUBMISSION=True after reviewing validation results."
    )